# SYMFLUENCE Notebook Path Setup Template

This template demonstrates the env-aware path configuration pattern used in the example notebooks.

Behavior:
- If SYMFLUENCE is installed from a cloned repository, detect the repo root from the installed package location and use:
  - `code_dir = repo_root`
  - `data_dir = repo_root.parent / 'SYMFLUENCE_data'`
- Otherwise, fall back to JupyterHub / packaged-install behavior:
  - `code_dir = $SYMFLUENCE_CODE_DIR` if set, else `~/SYMFLUENCE`
  - `data_dir = $SYMFLUENCE_DATA_DIR` if set, else `~/SYMFLUENCE_data`

The template also checks that `data_dir` exists and warns if `data_dir / 'installs'` is missing.


In [ ]:
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings('ignore', message='.*is an EXPERIMENTAL module.*')
warnings.filterwarnings('ignore', message='.*import failed.*')

print(f"Python executable: {sys.executable}")

try:
    import symfluence as symfluence_pkg
    print(f"SYMFLUENCE version: {symfluence_pkg.__version__}")
    print(f"SYMFLUENCE location: {Path(symfluence_pkg.__file__).parent}")
except ImportError:
    print("ERROR: SYMFLUENCE not found. Please activate the symfluence environment.")
    raise

def resolve_symfluence_paths():
    """Resolve SYMFLUENCE code/data paths for repo based and JupyterHub Docker image based symfluence installs."""
    pkg_dir = Path(symfluence_pkg.__file__).resolve().parent
    repo_root = None

    for candidate in [pkg_dir, *pkg_dir.parents]:
        if (
            (candidate / 'src' / 'symfluence').exists()
            or (candidate / '.git').exists()
            or (candidate / 'pyproject.toml').exists()
        ):
            repo_root = candidate.resolve()
            break

    if repo_root is not None:
        code_dir = repo_root
        data_dir = repo_root.parent / 'SYMFLUENCE_data'
        mode = 'symfluence repository detected'
    else:
        code_dir = Path(os.environ.get("SYMFLUENCE_CODE_DIR", str(Path.home() / "SYMFLUENCE")))
        data_dir = Path(os.environ.get("SYMFLUENCE_DATA_DIR", str(Path.home() / "SYMFLUENCE_data")))
        mode = 'installed package / JupyterHub fallback'

    return code_dir, data_dir, mode

current_dir = Path.cwd()
print(f"INITIAL working directory: {current_dir}")

if '.ipynb_checkpoints' in str(current_dir):
    current_dir = current_dir.parent
    os.chdir(current_dir)
    print(f"CHANGED to: {Path.cwd()}")
else:
    print("Working directory is correct")

code_dir, data_dir, path_mode = resolve_symfluence_paths()

if not data_dir.exists():
    raise RuntimeError(
        f"SYMFLUENCE data directory does not exist: {data_dir}. "
        "Set SYMFLUENCE_DATA_DIR or create the directory before continuing."
    )

installs_dir = data_dir / 'installs'
if not installs_dir.exists():
    print(
        f"WARNING: installs directory not found at {installs_dir}. "
        "If using the JupyterHub/Docker image, ensure your installs symlink is configured."
    )

print(f"Notebook directory: {current_dir}")
print(f"SYMFLUENCE path resolution mode: {path_mode}")
print(f"SYMFLUENCE data directory: {data_dir}")
print(f"SYMFLUENCE code directory: {code_dir}")
print(f"SYMFLUENCE installs directory: {installs_dir}")


## Example usage in a notebook

Use the resolved paths when creating a configuration:


In [ ]:
from symfluence.core.config.models import SymfluenceConfig

config = SymfluenceConfig.from_minimal(
    domain_name='example_domain',
    experiment_id='run_1',
    SYMFLUENCE_DATA_DIR=str(data_dir),
    SYMFLUENCE_CODE_DIR=str(code_dir),
    model='SUMMA',
    forcing_dataset='ERA5',
    definition_method='point',
    discretization='GRUs',
    pour_point_coords='46.78/-121.75',
    bounding_box_coords='46.781/-121.751/46.779/-121.749',
    time_start='2000-01-01 01:00',
    time_end='2000-12-31 23:00',
    spinup_period='2000-01-01, 2000-03-31',
    calibration_period='2000-04-01, 2000-09-30',
    evaluation_period='2000-10-01, 2000-12-31',
)

print('Config created successfully')
print(f'Config data_dir: {config.system.data_dir}')
print(f'Config code_dir: {config.system.code_dir}')
